In [ ]:
model_id = "distilbert/distilbert-base-german-cased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def compute_metrics(p: EvalPrediction):
    predictions = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    labels = p.label_ids

    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))

    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= 0.5)] = 1

    f1_micro = f1_score(y_true=labels, y_pred=y_pred, average='micro')
    f1_macro = f1_score(y_true=labels, y_pred=y_pred, average='macro', zero_division=0)
    precision = precision_score(y_true=labels, y_pred=y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true=labels, y_pred=y_pred, average='macro', zero_division=0)
    accuracy = accuracy_score(y_true=labels, y_pred=y_pred)

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall
    }

# Function to initialize DistilBERT with correct dropout parameters
def get_fresh_model(num_labels):
    return AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=num_labels,
        problem_type="multi_label_classification",
        dropout=0.10,             # DistilBERT hidden/encoder dropout
        attention_dropout=0.10,   # DistilBERT attention dropout
        seq_classif_dropout=0.20  # DistilBERT classification head dropout
    )

# ==========================================
# Part 1: Volltextannotation
# ==========================================
df1 = pd.read_excel('ground_truth_filtered.xlsx')
text_col = 'Headline_GER'
label_cols = [col for col in df1.columns if col != text_col]
df1['labels'] = df1[label_cols].values.astype(np.float32).tolist()
df1_clean = df1[[text_col, 'labels']].copy()

train_df1, rest_df1 = train_test_split(df1_clean, test_size=0.20, random_state=42)
dev_df1, test_df1 = train_test_split(rest_df1, test_size=0.40, random_state=42)

dataset1 = DatasetDict({
    'train': Dataset.from_pandas(train_df1, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df1, preserve_index=False),
    'test': Dataset.from_pandas(test_df1, preserve_index=False)
})

def tokenize_function(examples):
    return tokenizer(examples[text_col], padding="max_length", truncation=True, max_length=128)

tokenized_ds1 = dataset1.map(tokenize_function, batched=True)

model1 = get_fresh_model(num_labels=len(label_cols))

args1 = TrainingArguments(
    output_dir="./distilbert_volltext_finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.05,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir='./logs_volltext',
    logging_steps=10
)

trainer1 = Trainer(
    model=model1,
    args=args1,
    train_dataset=tokenized_ds1["train"],
    eval_dataset=tokenized_ds1["dev"],
    compute_metrics=compute_metrics
)

trainer1.train()
print("\n--- Part 1 Test Set Evaluation ---")
print(trainer1.evaluate(tokenized_ds1["test"]))


# ==========================================
# Part 2: Titelannotation
# ==========================================
df2 = pd.read_excel('ground_truth_LLM804.xlsx')
label_cols2 = [col for col in df2.columns if col != text_col]
df2['labels'] = df2[label_cols2].values.astype(np.float32).tolist()
df2_clean = df2[[text_col, 'labels']].copy()

train_df2, rest_df2 = train_test_split(df2_clean, test_size=0.20, random_state=42)
dev_df2, test_df2 = train_test_split(rest_df2, test_size=0.40, random_state=42)

dataset2 = DatasetDict({
    'train': Dataset.from_pandas(train_df2, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df2, preserve_index=False),
    'test': Dataset.from_pandas(test_df2, preserve_index=False)
})

tokenized_ds2 = dataset2.map(tokenize_function, batched=True)

# Instantiating a fresh base model so training starts clean
model2 = get_fresh_model(num_labels=len(label_cols2))

args2 = TrainingArguments(
    output_dir="./distilbert_titel_finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.05,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir='./logs_titel',
    logging_steps=10
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=tokenized_ds2["train"],
    eval_dataset=tokenized_ds2["dev"],
    compute_metrics=compute_metrics
)

trainer2.train()
print("\n--- Part 2 Test Set Evaluation ---")
print(trainer2.evaluate(tokenized_ds2["test"]))